# BLOCKv0.6 Lyco Skin Generator **v5** (fixed)

## v4 为什么失败
v4 跑完显示 `All done!`，但三张 64×64 皮肤**全是纯白空图**
（实测：均值 254.1，标准差 0.3，只有 6 种颜色）。

原因在推理参数：

| 参数 | v4 | 模型卡推荐 | 影响 |
|---|---|---|---|
| `prompt` | `""` | 完整提示词 | 无文本条件 |
| `guidance_scale` | **1.0** | **4.0** | **等于关掉 CFG 引导 → 输出塌成空白** |
| `num_inference_steps` | 20 | 30 | 细节不足 |

## v5 改了什么
1. 用模型卡的完整提示词
2. **`guidance_scale = 4.0`**
3. `num_inference_steps = 30`
4. 加**空图自动检测**（再出空白会立刻报警，不会假装成功）
5. 加**「第二层全白」修复** —— 模型输出 RGB 没有 alpha，把本该透明的
   外套层填成了白色；而 Minecraft 的第二层是盖在第一层上面渲染的，
   **全白 = 角色在游戏里是一坨白块**
6. 保存 RGBA + 拼合预览，便于人眼判断，而不是只存一张看不懂的 UV 图


In [ ]:
!pip install -q diffusers transformers accelerate pillow

In [ ]:
import os, io, json, time
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import numpy as np
import torch
from PIL import Image
import httpx
from diffusers import Flux2KleinPipeline

print(f"GPU          : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"GPU memory   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"torch        : {torch.__version__}")
import diffusers; print(f"diffusers    : {diffusers.__version__}")

# T4 是 Turing 架构，不支持 bfloat16 → 必须用 float16
DTYPE = torch.float16

# 可选：在 Kaggle 的 Add-ons > Secrets 里加 HF_TOKEN，能显著加快 15GB 权重的下载
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF_TOKEN   : 已加载")
except Exception:
    print("HF_TOKEN   : 无（下载会慢一些，但能跑）")


In [ ]:
MODEL_ID = "AliceKJ/BLOCKv0.6"
t0 = time.time()
print(f"加载 {MODEL_ID} （约 15 GB，用 CPU offload 塞进 15.6 GB 显存）...")

pipe = Flux2KleinPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    use_safetensors=True,
)
# 15.6 GB 显存装不下整个 4B 模型，必须开 CPU offload
pipe.enable_model_cpu_offload()
print(f"加载完成  {time.time()-t0:.0f}s")


In [ ]:
PROMPT = """Image-to-image translation using the reference image. The reference shows the same 3D Minecraft character with front and back views in a single image. Generate the corresponding Minecraft skin UV atlas in 64x64 pixel-art UV layout. High-quality anime-style. Flat shading, sharp pixel edges, no blur, no anti-aliasing. Keep consistent UV placement and mapping; match the same character design from the reference. Model type: classic (auto-detected Minecraft player model)."""

# 预览图：优先用我们自己的数据集，失败则回退到模型自带的示例
CANDIDATES = [
    # ★ 必须是公开且**存在**的数据集。实测 lilyco4242/lyco-preview 是 404，
    #   导致 v4/v5 一直静默回退到模型自带示例，生成的从来不是 Lyco。
    "https://huggingface.co/datasets/lyco42/lyco-preview/resolve/main/lyco_preview.png",
    "https://huggingface.co/AliceKJ/BLOCKv0.6/resolve/main/examples/ex1_preview.png",
]

preview_image = None
for url in CANDIDATES:
    try:
        r = httpx.get(url, timeout=90, follow_redirects=True)
        r.raise_for_status()
        im = Image.open(io.BytesIO(r.content)).convert("RGB")
        arr = np.asarray(im)
        preview_image = im
        LAST_URL = url
        print(f"预览图 OK  {im.size}  均值={arr.mean():.1f} 标准差={arr.std():.1f}")
        print(f"  来源: {url}")
        break
    except Exception as e:
        print(f"预览图失败 {type(e).__name__}: {str(e)[:90]}  <- {url}")

assert preview_image is not None, "拿不到预览图"
preview_image


In [ ]:
# ★ 防呆：确认拿到的**确实是 Lyco 的预览图**，不是回退到别人的示例
assert preview_image is not None, "拿不到预览图"

USED_FALLBACK = "AliceKJ/BLOCKv0.6" in str(globals().get("LAST_URL", ""))
if "LAST_URL" in globals() and USED_FALLBACK:
    print()
    print("!!! 警告：用的是模型自带的示例图，生成的不会是 Lyco !!!")
    print("!!! 请检查 lilyco4242/lyco-preview 数据集是否为公开 !!!")
else:
    print("OK：用的是我们自己的 Lyco 预览图")

# 记录实际用的 URL，供上面判断
print("实际使用:", globals().get("LAST_URL", "(未记录)"))

preview_image


In [ ]:
# ── 第二层（外套层）在 64x64 UV 里的位置 ──
OVERLAY_RECTS = {
    "hat":     (32, 0, 64, 16),
    "jacket":  (16, 32, 40, 48),
    "rSleeve": (40, 32, 56, 48),
    "lSleeve": (48, 48, 64, 64),
    "rPant":   (0, 32, 16, 48),
    "lPant":   (0, 48, 16, 64),
}
# 第一层（本体）—— 绝不能动。注意左臂主层是 (32,48)-(48,64)，不是到 64
FIRST_LAYER_RECTS = [
    (0, 0, 32, 16), (16, 16, 40, 32), (40, 16, 56, 32),
    (0, 16, 16, 32), (16, 48, 32, 64), (32, 48, 48, 64),
]


def fix_white_overlay(img, white_thresh=235, keep_ratio=0.12, verbose=True):
    """把第二层里的近白像素判为背景、置为透明。

    模型输出是 RGB 没有 alpha，它把「应该透明」的外套层填成了白色
    （因为训练用的预览图背景就是白的）。而 MC 的第二层盖在第一层上面渲染，
    全白 = 角色变成一坨白块。
    """
    a = np.asarray(img.convert("RGBA")).copy()
    for name, (x0, y0, x1, y1) in OVERLAY_RECTS.items():
        sub = a[y0:y1, x0:x1]
        rgb = sub[:, :, :3].astype(np.int16)
        is_white = (rgb > white_thresh).all(axis=2)
        before = int((sub[:, :, 3] > 0).sum())
        sub[:, :, 3][is_white] = 0
        after = int((sub[:, :, 3] > 0).sum())
        if after < sub[:, :, 3].size * keep_ratio:
            sub[:, :, 3] = 0
            if verbose: print(f"    {name:<8} 清 {before:>3}px 白 -> 整层清空")
        elif verbose:
            print(f"    {name:<8} 清 {before-after:>3}px 白 -> 保留 {after:>3}px")
    # 第一层被误伤了没？
    broken = [r for r in FIRST_LAYER_RECTS if (a[r[1]:r[3], r[0]:r[2], 3] < 255).any()]
    return Image.fromarray(a, "RGBA"), (len(broken) == 0)


def assemble(skin, view="front"):
    """按 UV 拼成人形，人眼才能判断好不好"""
    g = skin.load()
    out = Image.new("RGBA", (16, 32), (0, 0, 0, 0))
    o = out.load()

    def blit(box, dx, dy):
        x0, y0, x1, y1 = box
        for y in range(y0, y1):
            for x in range(x0, x1):
                tx, ty = dx + (x - x0), dy + (y - y0)
                if 0 <= tx < 16 and 0 <= ty < 32:
                    o[tx, ty] = g[x, y]

    if view == "front":
        blit((8, 8, 16, 16), 4, 0); blit((40, 20, 44, 32), 0, 8)
        blit((20, 20, 28, 32), 4, 8); blit((36, 52, 40, 64), 12, 8)
        blit((4, 20, 8, 32), 4, 20); blit((20, 52, 24, 64), 8, 20)
    else:
        blit((24, 8, 32, 16), 4, 0); blit((52, 20, 56, 32), 0, 8)
        blit((32, 20, 40, 32), 4, 8); blit((44, 52, 48, 64), 12, 8)
        blit((12, 20, 16, 32), 4, 20); blit((28, 52, 32, 64), 8, 20)
    return out


print("工具函数就绪")


In [ ]:
def generate_skin(seed, steps=30, guidance=4.0):
    """★ 关键：guidance 必须是 4.0。v4 用 1.0 等于关掉 CFG，输出直接塌成空白。"""
    gen = torch.Generator("cpu").manual_seed(seed)
    t = time.time()
    result = pipe(
        prompt=PROMPT,
        image=preview_image.convert('RGB').resize((512, 512)),  # ★ v5 漏了这句：
        #   不 resize 会按 1024 推理，慢 4 倍且输出尺寸不对

        num_inference_steps=steps,
        guidance_scale=guidance,
        generator=gen,
    ).images[0]
    dt = time.time() - t

    raw = result.convert("RGB")
    arr = np.asarray(raw)
    # ★ 空图检测：v4 就是这样悄悄失败的
    blank = (arr.std() < 3.0) or (len(np.unique(arr.reshape(-1, 3), axis=0)) < 8)
    print(f"  seed={seed:<6} {dt:5.1f}s  size={raw.size}  均值={arr.mean():6.1f} "
          f"标准差={arr.std():5.1f}  {'❌ 空白图!' if blank else '✅ 有内容'}")
    return raw, blank


print("=== 生成 ===")
SEEDS = [42, 123, 2024, 7]
raws = {}
for s in SEEDS:
    torch.cuda.empty_cache()
    raws[s] = generate_skin(s)

n_blank = sum(1 for _, b in raws.values() if b)
print()
print(f"结果: {len(SEEDS)-n_blank}/{len(SEEDS)} 张有内容" +
      ("   ← 还有空白图，检查 guidance_scale" if n_blank else "   ✅ 全部正常"))


In [ ]:
FINAL = {}
for s, (raw, blank) in raws.items():
    if blank:
        print(f"seed {s}: 跳过（空白图）")
        continue
    skin64 = raw.resize((64, 64), Image.NEAREST)
    print(f"seed {s}:")
    fixed, ok = fix_white_overlay(skin64)
    print(f"    第一层完整性: {'✅ 未受影响' if ok else '❌ 被破坏'}")
    p = f"/kaggle/working/lyco_skin_seed{s}.png"
    fixed.save(p)
    FINAL[s] = fixed
    print(f"    -> {p}")

print()
print(f"共保存 {len(FINAL)} 张皮肤")


In [ ]:
if FINAL:
    Z = 8
    tiles = []
    for s, sk in FINAL.items():
        tiles.append(sk.resize((64 * Z, 64 * Z), Image.NEAREST))
        for v in ("front", "back"):
            tiles.append(assemble(sk, v).resize((16 * Z, 32 * Z), Image.NEAREST))

    gap = 14
    W = sum(t.width for t in tiles) + gap * (len(tiles) - 1)
    H = max(t.height for t in tiles)
    sheet = Image.new("RGBA", (W, H), (24, 24, 28, 255))
    x = 0
    for t in tiles:
        sheet.alpha_composite(t, (x, 0))
        x += t.width + gap
    sheet.convert("RGB").save("/kaggle/working/lyco_skins_preview.png")
    print("预览图 -> /kaggle/working/lyco_skins_preview.png")
    print("顺序: 每个 seed 的 图集 + 正面 + 背面")
    sheet.convert("RGB")
else:
    print("没有产出任何皮肤 —— 回到上一格看空图告警")
